In [6]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import scipy
import pandas as pd
import tifffile as tif #211108 update, the latest skimage does not have external anymore
import seaborn as sns
from scipy.stats import lognorm, nbinom
from PIL import Image
import skimage
import cv2
from scipy.ndimage import maximum_filter, label
from scipy.optimize import curve_fit
import skimage.io
from scipy.io import savemat
from skimage import img_as_ubyte


T:\Anaconda\Anaconda\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [2]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['xtick.major.size'] = 3
plt.rcParams['xtick.major.width'] = 1
plt.rcParams['xtick.bottom'] = True
plt.rcParams['ytick.left'] = True
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['legend.edgecolor'] = 'w'


In [ ]:
def rotave_mass_3d(volume):
    """
    Compute the radial intensity of a cubic volume relative to its center.
    
    Parameters:
        volume (np.ndarray): 3D cubic array.
        
    Returns:
        np.ndarray: 1D array with the accumulated intensities for each radial bin,
                    excluding the center bin.
    """
    N = volume.shape[0]
    half = (N - 1) // 2
    # Create a coordinate grid from -half to half in x, y and z
    x = np.arange(-half, half + 1)
    y = np.arange(-half, half + 1)
    z = np.arange(-half, half + 1)
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    # Compute radial distances from the center (spherical radius)
    rho = np.sqrt(X**2 + Y**2 + Z**2)
    # Bin index: equivalent to 1 + floor(rho+0.5) in Matlab
    pix = 1 + np.floor(rho + 0.5).astype(np.int64)
    # Use np.bincount (with 0-index correction) to accumulate intensities in spherical shells
    counts = np.bincount(pix.ravel() - 1, weights=volume.ravel())
    # Return bins 2 to half+1 (excluding the very center)
    return counts[1:half + 1]

def domain_ms_3d(domain, cw, rn=0):
    """
    Calculate the domain mass scaling ring, mass scaling and weight for a 3D volume.
    
    Parameters:
        domain (np.ndarray): A cubic 3D image.
        cw (int): Half–size of the center window (in voxels).
        rn (int): Number of random centers to use in the domain (default 0 means use all).
    
    Returns:
        MSRing (np.ndarray): 2D array (radial bins x centers) of mass scaling ring values.
        MS (np.ndarray): 2D array of cumulative mass scaling values.
        weight (np.ndarray): 1D array with weight values for each center.
    """
    N = domain.shape[0]
    hw = (N - 1) // 2

    # Extract the center subvolume.
    # MATLAB equivalent: center = domain(hw+1-cw:end-hw+cw, hw+1-cw:end-hw+cw, hw+1-cw:end-hw+cw);
    # start = (hw + 1 - cw) - 1  # adjust for 0-indexing
    # stop = N - hw + cw   # stop is exclusive in Python
    # center = domain[start:stop, start:stop, start:stop]
    
    # Find indices of nonzero voxels in the center and randomize them.
    # centers_idx = np.flatnonzero(center)
    # centers_idx_rd = np.random.permutation(centers_idx)
    # if rn != 0: #if specified, e.g. rn = 3, only take 3 centers. Otherwise, by default, take 11*11*11 centers
    #     rn = min(rn, len(centers_idx_rd))
    #     centers_idx_rd = centers_idx_rd[:rn]
    
    # weight = np.zeros(len(centers_idx_rd))
    # The number of radial bins is (hw - cw)
    # MSRing = np.zeros(hw - cw, len(centers_idx_rd)))
    MSRing = np.zeros(hw - cw)

    # for index in np.indices(center.shape).reshape(len(center.shape), -1).T:
    #     if index[2] == 10:
    #         print("Processing index " + str(index))
            
    #     in_volume = domain[ index[0] : index[0] + 2 * (hw - cw) + 1, index[1] : index[1] + 2 * (hw - cw) + 1, index[2] : index[2] + 2 * (hw - cw) + 1]
    #     MSRing[:,center.shape[0]**2 * index[0] + center.shape[0] * index[1] + index[2]] = rotave_mass_3d(in_volume)
    #     weight[center.shape[0]**2 * index[0] + center.shape[0] * index[1] + index[2]] = center[index[0], index[1], index[2]]

    # for index in np.indices(center.shape).reshape(len(center.shape), -1).T:
    #     if index[2] == 10:
    #         print("Processing index " + str(index))
            
    #     in_volume = domain[ index[0] : index[0] + 2 * (hw - cw) + 1, index[1] : index[1] + 2 * (hw - cw) + 1, index[2] : index[2] + 2 * (hw - cw) + 1]
    #     MSRing[:,center.shape[0]**2 * index[0] + center.shape[0] * index[1] + index[2]] = rotave_mass_3d(in_volume)
    #     weight[center.shape[0]**2 * index[0] + center.shape[0] * index[1] + index[2]] = center[index[0], index[1], index[2]]
    index = [cw,cw,cw]
    in_volume = domain[ index[0] : index[0] + 2 * (hw - cw) + 1, index[1] : index[1] + 2 * (hw - cw) + 1, index[2] : index[2] + 2 * (hw - cw) + 1]
    in_volume = in_volume/in_volume.max()
    print(in_volume.max())
    MSRing = rotave_mass_3d(in_volume)
    weight = 1
    
    
    # Cumulative sum (similar to MATLAB’s for-loop cumulative sum)
    MS = np.cumsum(MSRing, axis=0)
    # Compute normalization using a volume of ones with the same shape as in_vol.
    sampled = rotave_mass_3d(np.ones_like(in_volume))
    sampled_cum = np.cumsum(sampled)
    
    # Normalize the mass scaling ring and the cumulative mass scaling.
    # MSRing_norm = MSRing / sampled[:, np.newaxis]
    # MS_norm = MS / sampled_cum[:, np.newaxis]
    MSRing_norm = MSRing / sampled
    MS_norm = MS / sampled_cum
    
    return MSRing_norm, MS_norm, weight

def linear_model(x, p1, p2):
    """Linear model: f(x) = p1 * x + p2."""
    return p1 * x + p2

def create_fit_ignore_inf_value(x, y):
    """
    Fit a linear model (y = p1*x + p2) to data (x,y) with p1 constrained to be ≤ 3.
    
    Returns a dictionary with fitted parameters and a goodness-of-fit dictionary.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    lower_bounds = (-np.inf, -np.inf)
    upper_bounds = (3, np.inf)
    popt, _ = curve_fit(linear_model, x, y, bounds=(lower_bounds, upper_bounds))
    y_fit = linear_model(x, *popt)
    residuals = y - y_fit
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 1.0
    rmse = np.sqrt(np.mean(residuals**2))
    gof = {'rsquared': r_squared, 'rmse': rmse}
    fitresult = {'p1': popt[0], 'p2': popt[1]}
    return fitresult, gof

def moving_window_slope(x, y, windowsize=5):
    """
    Computes a moving-window slope of y versus x.
    
    Parameters:
        x (np.ndarray): 1D array of independent variable values.
        y (np.ndarray): 1D array of dependent variable values.
        windowsize (int, optional): Window size for computing slopes (default 5).
    
    Returns:
        r (np.ndarray): x values corresponding to the center of each window.
        slope (np.ndarray): Computed slope for each window.
    """
    mask = ~np.isnan(y) & ~np.isinf(y)
    x = x[mask]
    y = y[mask]
    slopes = []
    for i in range(0, len(y) - windowsize - 1):
        xmov = x[i: i + windowsize + 1]
        ymov = y[i: i + windowsize + 1]
        s = create_fit_ignore_inf_value(xmov, ymov)
        slopes.append(s[0]['p1'])
    slopes = np.array(slopes)
    start_index = int(np.floor(windowsize / 2))
    end_index = start_index + len(slopes)
    r = x[start_index:end_index]
    return r, slopes

In [ ]:
# --- Main Loop for 3D processing ---

# Set the directory where results will be saved.
savedir = "\\"

# Read the 3D volume. (Replace the file path as needed.)
CTRL_volume = tif.imread('\\Nucleosome_Gaussian_rad5.tif')
# Load domain center data
dc = np.array([(198, 147, 121), (367, 29, 185), (402, 296, 125), (316, 461, 49), (540, 494, 112), (345, 225, 432), (488, 214, 332), (86, 471, 73), (69, 376, 483), (468, 298, 172), (83, 464, 376), (164, 30, 482), (347, 198, 369), (306, 217, 562), (541, 109, 112), (494, 70, 95), (159, 534, 421), (107, 244, 515), (57, 273, 38), (543, 536, 442), (396, 431, 444), (283, 291, 128), (226, 211, 395), (540, 260, 34), (400, 179, 137), (367, 295, 235), (439, 145, 323), (321, 245, 49), (395, 54, 245), (171, 122, 394), (542, 390, 148), (58, 161, 33), (172, 122, 339), (508, 438, 380), (237, 552, 27), (384, 465, 89), (74, 252, 136), (407, 476, 444), (446, 500, 201), (474, 423, 447), (504, 416, 75), (443, 354, 280), (508, 277, 370), (440, 470, 507), (455, 197, 89), (462, 458, 558), (245, 381, 435), (454, 213, 514), (215, 26, 423), (135, 296, 51), (346, 308, 483), (316, 114, 529), (494, 69, 330), (499, 33, 150), (418, 214, 419), (74, 522, 377), (431, 384, 530), (407, 510, 64), (378, 422, 496), (520, 274, 159), (108, 208, 161), (440, 259, 242), (392, 293, 205), (124, 411, 512), (479, 353, 381), (531, 149, 386), (542, 337, 181), (147, 87, 338), (81, 198, 45), (309, 77, 264), (453, 243, 576), (462, 390, 327), (157, 55, 26), (206, 487, 23), (330, 189, 152), (406, 251, 493), (372, 203, 66), (467, 437, 147), (122, 299, 386), (504, 488, 321), (139, 348, 404), (363, 80, 206), (23, 163, 132), (448, 308, 545), (455, 533, 536), (517, 476, 266), (551, 36, 535), (570, 150, 106), (286, 544, 330), (462, 388, 426), (318, 520, 72), (357, 318, 112), (509, 197, 192), (421, 88, 152), (225, 362, 569), (542, 144, 327), (526, 279, 500), (426, 550, 513), (575, 57, 514), (523, 203, 426), (429, 134, 34), (159, 107, 59), (362, 230, 520), (285, 242, 505), (526, 517, 506), (254, 42, 71), (480, 363, 290), (414, 272, 419), (418, 423, 371), (472, 354, 37), (109, 534, 346), (422, 383, 249), (62, 524, 315), (64, 566, 358), (421, 268, 367), (382, 328, 204), (38, 242, 257), (438, 355, 347), (361, 530, 88), (357, 500, 475), (323, 107, 82), (476, 21, 51), (334, 246, 198), (462, 138, 223), (473, 296, 282), (276, 209, 199), (38, 375, 282), (325, 525, 447), (27, 92, 168), (360, 58, 307), (78, 74, 122), (84, 175, 490), (454, 117, 198), (180, 249, 505), (318, 253, 396), (57, 152, 476), (408, 371, 190), (568, 44, 85), (297, 257, 218), (329, 123, 499), (545, 459, 54), (380, 151, 227), (382, 484, 261), (481, 308, 93), (304, 223, 249), (482, 395, 42), (89, 295, 324), (433, 137, 267), (557, 304, 128), (202, 295, 115), (301, 27, 78), (359, 383, 24), (134, 93, 579), (291, 32, 454), (131, 67, 449), (190, 301, 533), (174, 176, 50), (520, 467, 205), (230, 148, 40), (378, 186, 172), (343, 433, 465), (500, 462, 521), (171, 324, 67), (383, 355, 478), (438, 433, 553), (367, 171, 429), (477, 283, 91), (467, 388, 259), (466, 116, 358), (424, 161, 198), (487, 375, 520), (363, 353, 217), (541, 237, 518), (433, 69, 255), (346, 98, 33), (513, 243, 255), (240, 220, 233), (203, 276, 472), (121, 450, 408), (413, 181, 250), (399, 159, 63), (389, 233, 239), (371, 194, 472), (29, 548, 280), (340, 541, 528), (155, 154, 26), (479, 568, 147), (219, 53, 143), (528, 278, 401), (382, 412, 37), (477, 506, 105), (333, 188, 432), (402, 437, 291), (474, 129, 281), (76, 426, 446), (66, 46, 94), (310, 445, 499), (554, 577, 433), (487, 271, 258), (292, 29, 235), (30, 228, 469), (432, 135, 572), (153, 221, 69), (461, 419, 109), (82, 287, 579), (338, 172, 264), (77, 238, 229), (109, 195, 376), (57, 52, 401), (465, 512, 509), (437, 508, 92), (501, 157, 119), (211, 72, 52), (208, 90, 395), (499, 371, 132), (77, 321, 38), (569, 284, 572), (337, 554, 318), (541, 502, 369), (434, 386, 147), (39, 502, 359), (459, 503, 136), (321, 205, 66), (553, 312, 258), (299, 372, 100), (481, 225, 546), (272, 323, 562), (134, 525, 488), (142, 483, 428), (349, 475, 537), (383, 568, 412), (81, 74, 389), (35, 81, 495), (157, 513, 22), (455, 402, 226), (125, 390, 560), (368, 146, 31), (536, 503, 212), (490, 344, 575), (427, 564, 289), (20, 512, 305), (52, 132, 513), (507, 383, 460), (115, 192, 116), (480, 74, 551), (567, 517, 389), (405, 30, 422), (465, 409, 286), (504, 287, 428), (300, 522, 48), (187, 476, 486), (548, 558, 495), (470, 573, 484), (184, 147, 420), (508, 350, 42), (43, 374, 81), (374, 423, 320), (261, 284, 76), (465, 72, 384), (46, 276, 561), (385, 218, 31), (35, 255, 568), (100, 212, 473), (153, 145, 185), (461, 521, 164), (367, 191, 123), (376, 123, 298), (334, 211, 212), (213, 319, 89), (399, 333, 554), (395, 220, 330), (50, 347, 386), (96, 525, 253), (219, 388, 486), (136, 550, 39), (346, 535, 60), (451, 456, 76), (487, 486, 451), (481, 451, 219), (146, 545, 540), (37, 570, 104), (454, 561, 121), (124, 365, 575), (370, 275, 549), (490, 213, 89), (93, 379, 432), (37, 36, 55), (368, 202, 274), (357, 152, 317), (408, 412, 526), (98, 386, 566), (159, 143, 212), (329, 417, 67), (551, 75, 519), (449, 240, 293), (408, 485, 216), (405, 334, 247), (239, 307, 102), (198, 117, 47), (409, 181, 90), (336, 147, 48), (417, 191, 463), (187, 529, 579), (374, 414, 110), (399, 466, 284), (511, 535, 289), (551, 183, 106), (442, 188, 138), (269, 486, 507), (528, 304, 121), (398, 265, 285), (328, 188, 102), (327, 54, 552), (55, 200, 154), (442, 435, 191), (140, 29, 415), (420, 427, 79), (552, 519, 522), (276, 262, 87), (408, 478, 130), (307, 73, 58), (104, 59, 431), (299, 303, 401), (343, 248, 370), (445, 334, 97), (541, 101, 506), (437, 166, 88), (178, 247, 547), (571, 124, 34), (255, 319, 72), (560, 22, 161), (465, 204, 165), (86, 349, 359), (164, 63, 217), (352, 262, 138), (222, 44, 66), (337, 31, 566), (492, 198, 239), (528, 216, 45), (49, 183, 216), (433, 31, 369), (397, 380, 516), (371, 269, 306), (90, 243, 263), (439, 30, 543), (392, 84, 332), (236, 321, 40), (72, 529, 58), (332, 240, 106), (536, 338, 216), (217, 439, 465), (475, 139, 396), (433, 74, 527), (437, 419, 33), (30, 37, 313), (422, 462, 43), (490, 310, 456), (114, 246, 38), (109, 124, 32)])

ahw_thresh = 25
hw = 150
cw = 5
rn = 0
dr = 2
r = np.arange(dr, dr * hw + dr, 2)
sz = CTRL_volume.shape  # expected shape: (rows, cols, slices)

# Assume 'maxima' is a NumPy array of shape (n_centers, 3), where each row is [x, y, z].
# For example, you might have loaded it from a file.
# maxima = np.load('maxima.npy')

# for i in np.arange(3):
for i in np.arange(dc.shape[0]):
    
    # For 3D, assume centroid is given in (x, y, z) coordinates.
    centroid = [dc[i][0], dc[i][1], dc[i][2]]
    
    # Compute the minimum distance from the centroid to the image edges.
    # Note: In a 3D volume with shape (slices, rows, cols), we assume:
    # row corresponds to x, col to y, and slice to z.
    dist = min(centroid[0],
               centroid[1],
               centroid[2],
               sz[0] - 1 - centroid[0],
               sz[1] - 1 - centroid[1],
               sz[2] - 1 - centroid[2])
    
    ahw = min(dist, hw)
    if ahw < ahw_thresh:
        continue
    print("Processing domain " + str(i + 1))
    
    # Extract the domain subvolume from CTRL_volume.
    # Use: row (y), col (x), slice (z)
    row_start = int(centroid[1] - ahw)
    row_end   = int(centroid[1] + ahw + 1)
    col_start = int(centroid[2] - ahw)
    col_end   = int(centroid[2] + ahw + 1)
    slice_start = int(centroid[0] - ahw)
    slice_end   = int(centroid[0] + ahw + 1)
    domain = CTRL_volume[slice_start:slice_end, row_start:row_end, col_start:col_end] # domain size must be an odd number
    
    # Save a projection image of the domain (average projection along z).
    domain_proj = np.sum(domain, axis=0)
    # domain_proj_uint8 = img_as_ubyte(domain_proj)
    skimage.io.imsave(savedir + f"DomainProj{i+1:02d}.tif", domain_proj)
    
    # Select the first (ahw - cw) radial bins.
    r_domain = r[:int(ahw - cw)]
    MSRing, MS, weight = domain_ms_3d(domain, cw, rn)
    
    # --- Radial Density Calculation ---
    # meanRD = (MSRing @ weight) / np.sum(weight)
    # sdRD = np.sqrt((((MSRing - meanRD[:, np.newaxis])**2) @ weight) / ((len(weight) - 1) * np.mean(weight)))
    meanRD = MSRing
    sdRD = np.zeros(MSRing.shape)
    
    # --- Scale MSRing and MS ---
    # MSRing_scaled = MSRing * (r_domain[:, np.newaxis] * 2 * np.pi)
    # MS_scaled = MS * ((r_domain**2)[:, np.newaxis] * np.pi)
    MSRing_scaled = MSRing * ((r_domain**2) * 4 * np.pi)
    MS_scaled = MS * ((r_domain**3) * 4/3 * np.pi)
    
    # meanMSRing = (MSRing_scaled @ weight) / np.sum(weight)
    # sdMSRing = np.sqrt((((MSRing_scaled - meanMSRing[:, np.newaxis])**2) @ weight) /
    #                    ((len(weight) - 1) * np.mean(weight)))
    meanMSRing = MSRing_scaled
    sdMSRing = np.zeros(meanMSRing.shape)
    
    # meanMS = (MS_scaled @ weight) / np.sum(weight)
    # sdMS = np.sqrt((((MS_scaled - meanMS[:, np.newaxis])**2) @ weight) /
    #                ((len(weight) - 1) * np.mean(weight)))
    meanMS = MS_scaled
    sdMS = np.zeros(meanMS.shape)
    
    # --- Plotting ---
    plt.figure(1)
    plt.errorbar(r_domain, meanMSRing, yerr=sdMSRing, fmt='o-')
    plt.xscale('log')
    plt.yscale('log')
    plt.savefig(savedir + f"DomainMSRing{i+1:02d}.tif", format='tiff')
    plt.close(1)
    
    plt.figure(2)
    plt.errorbar(r_domain, meanMS, yerr=sdMS, fmt='o-')
    plt.xscale('log')
    plt.yscale('log')
    plt.savefig(savedir + f"DomainMS{i+1:02d}.tif", format='tiff')
    plt.close(2)
    
    logr = np.log(r_domain)
    logMS = np.log(meanMS)
    rslope, slope = moving_window_slope(logr, logMS)
    rslope = np.exp(rslope)
    plt.figure(3)
    plt.loglog(rslope, slope + 1, 'o-')
    plt.savefig(savedir + f"DomainMSFirstDev{i+1:02d}.tif", format='tiff')
    plt.close(3)
    
    plt.figure(4)
    plt.errorbar(r_domain, meanRD, yerr=sdRD, fmt='o-')
    plt.savefig(savedir + f"RadialDensity{i+1:02d}.tif", format='tiff')
    plt.close(4)
    
    # --- Save variables to a .mat file ---
    mat_dict = {
        'meanMSRing': meanMSRing,
        'sdMSRing': sdMSRing,
        'meanMS': meanMS,
        'sdMS': sdMS,
        'rslope': rslope,
        'slope': slope,
        'meanRD': meanRD,
        'sdRD': sdRD,
        'r_domain': r_domain,
        'cw': cw,
        'rn': rn,
        'centroid': centroid
    }
    savemat(savedir + f"Domain{i+1:02d}.mat", mat_dict)